# Оценка RAG-пайплайна

In [1]:
import os
import warnings
import pandas as pd
from dotenv import load_dotenv
from datasets import Dataset

from openai import OpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import llm_factory
from ragas import evaluate

In [2]:
# RAGAS переходит на v1.0 так что игнорим deprecation
warnings.filterwarnings("ignore", category=DeprecationWarning)

from ragas.metrics import (
    ContextPrecision,
    ContextRecall,
    AnswerRelevancy,
    Faithfulness
)

load_dotenv()

True

In [3]:
# Инициализация API-клиента Cloud.ru
client = OpenAI(
    api_key=os.getenv('CLOUDRU_API_KEY'),
    base_url="https://foundation-models.api.cloud.ru/v1"
)

# LLM-as-a-Judge
llm = llm_factory(
    model="Qwen/Qwen3-235B-A22B-Instruct-2507", 
    provider="openai", 
    client=client, 
    max_tokens=16384
)

# Эмбеддинги
embeddings = HuggingFaceEmbeddings(
    model_name="Qwen/Qwen3-Embedding-0.6B"
)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [4]:
# Загрузка датасета
df = pd.read_parquet('../../data/eval/gd_filled.parquet')
dataset = Dataset.from_pandas(df)
print(f"Dataset loaded. Total samples: {len(dataset)}")

Dataset loaded. Total samples: 50


In [5]:
# Запуск оценки
result = evaluate(
    dataset=dataset,
    metrics=[
        ContextPrecision(llm=llm),
        ContextRecall(llm=llm),
        AnswerRelevancy(llm=llm, embeddings=embeddings),
        Faithfulness(llm=llm)
    ]
)

print(result)

Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

{'context_precision': 0.5642, 'context_recall': 0.7458, 'answer_relevancy': 0.8164, 'faithfulness': 0.8365}
